In [22]:
%pip install datasets
%pip install openai

from datasets import load_dataset
import requests
%pip install Pillow
from PIL import Image
import pandas as pd
from pathlib import Path
import json
%pip install python-dotenv
from dotenv import load_dotenv
load_dotenv()
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY is not set. Add it to your .env file before running this notebook.")

openai_client = OpenAI(api_key=api_key)


 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

^C
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/prathibhasrinivas/ironhack/.venv/lib/python3.14/site-packages/pip/__main__.py", line 24, in <module>
    sys.exit(_main())
  File "/Users/prathibhasrinivas/ironhack/.venv/lib/python3.14/site-packages/pip/_internal/cli/main.py", line 47, in main
    from pip._internal.cli.autocompletion import autocomplete
  File "/Users/prathibhasrinivas/ironhack/.venv/lib/python3.14/site-packages/pip/_internal/cli/autocompletion.py", line 12, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/Users/prathibhasrinivas/ironhack/.venv/lib/python3.14/site-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._vendor.rich.markup import escape
  File "/Users/prathibhasrinivas/ironhack/.venv/lib/python3.14/site-packages/pip/_vendor/rich/__init__.py", line 6, in <module>
    from ._extension import

In [ ]:
import base64

import base64
from io import BytesIO
from pathlib import Path

def encode_image_to_base64(image_value):
    """Encode either a file path or a PIL image object to base64."""
    if hasattr(image_value, "save"):
        buffer = BytesIO()
        image_value.save(buffer, format="JPEG")
        image_bytes = buffer.getvalue()
    elif isinstance(image_value, (str, Path)):
        image_path = Path(image_value)
        with image_path.open("rb") as img_file:
            image_bytes = img_file.read()
    else:
        raise TypeError(f"Unsupported image type: {type(image_value)!r}")

    return base64.b64encode(image_bytes).decode("utf-8")

# Example usage
sample_image = products_df.iloc[0]["image"]
encoded_image = encode_image_to_base64(sample_image)

print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")


In [ ]:
products_df


In [ ]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters

In [ ]:
import base64
from io import BytesIO
from pathlib import Path

def encode_image(image_value):
    if hasattr(image_value, "save"):
        buffer = BytesIO()
        image_value.save(buffer, format="JPEG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

    with Path(image_value).open("rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def call_openai(image_value, text_prompt):
    base64_image = encode_image(image_value)
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": text_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                    },
                ],
            }
        ],
    )
    return response

# Example usage with a dataframe image
image_row = products_df.iloc[0]
image_value = image_row.get("image", image_row.get("image_path"))
if image_value is None:
    raise KeyError("Could not find an image or image_path column in products_df")
text_prompt = "Generate a product listing for this image."

response = call_openai(image_value, text_prompt)
print(response.output_text)

### Product Listing: Men’s Casual Checkered Shirt

**Product Name:** Stylish Men’s Casual Checkered Shirt

**Description:**
Elevate your wardrobe with our trendy men’s checkered shirt, perfect for both casual outings and smart-casual occasions. This shirt features a classic check pattern in navy, white, and burgundy, providing a stylish look that’s easy to pair with jeans or chinos. The short sleeves make it ideal for warm weather, while the modern collar adds a touch of sophistication.

**Key Features:**
- **Material:** Soft, breathable cotton for all-day comfort
- **Design:** Stylish checkered pattern
- **Fit:** Regular fit, designed for ease of movement
- **Sleeves:** Short sleeves for a relaxed look
- **Occasion:** Perfect for casual wear, parties, or office settings

**Available Sizes:** S, M, L, XL

**Care Instructions:**
- Machine wash cold
- Do not bleach
- Tumble dry low
- Iron on low heat

**Price:** $39.99

**Customer Reviews:**
⭐️⭐️⭐️⭐️⭐️ - "Great fit and super comfortable!